In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Limpieza y Deduplicación -> Capa Silver
Aplicación de reglas de limpieza y guardado en formato Delta.

In [0]:
df_user = spark.table('castor.bronze.brz_dim_users')

In [0]:
display(df_user)

In [0]:
# Limpieza usuarios
# Análisis y normalización de fecha_registro

# Usamos F.expr() para llamar a try_to_date de SQL, que devuelve NULL si el formato no coincide
df_user = df_user.withColumn(
    'fecha_registro',
    F.coalesce(
        F.expr("try_to_date(fecha_registro, 'yyyy-MM-dd')"),
        F.expr("try_to_date(fecha_registro, 'dd/MM/yyyy')"),
        F.expr("try_to_date(fecha_registro, 'yyyy-MM-dd HH:mm:ss')")
    )
)

# Deduplicar por id_usuario tomando el más reciente (si hubiera un timestamp, aquí se usa row_number())
window = Window.partitionBy('id_usuario').orderBy(F.col('fecha_registro').desc())
df_user_clean = df_user.withColumn('rn', F.row_number().over(window)).filter(F.col('rn') == 1).drop('rn')

# Imputación de nulos según README
df_user_clean = df_user_clean.fillna({'pais': 'Desconocido'})
mediana_edad = df_user_clean.approxQuantile('edad', [0.5], 0.01)[0]
df_user_clean = df_user_clean.fillna({'edad': mediana_edad})

display(df_user_clean)

In [0]:
df_user_clean.write.mode('overwrite').saveAsTable('castor.silver.slv_dim_users')
display(df_user_clean)